# Matrix multiplication with the IME

The RISC-V integrated matrix extension adds fused matrix-dot instructions:
`vmadot` for integers and `vfmadot` for fp16. With `TRITON_RISCV_USE_IME=1` the
backend lowers `linalg.matmul` to `ime.vmadot` / `ime.vfmadot` using buddy-mlir
passes and cross-compiles the result with the `+xsmtime` feature, so it emits
RISC-V code no matter which host you run this on.

The flag is read while a kernel is compiled and, because it is not part of
Triton's cache key, the two variants below are written as two separate kernels
so that both of them really do get compiled.


## Activate the backend

triton-riscv installs itself as a Triton plugin under the name
`triton_shared`. On a CPU-only host Triton picks it on its own, but saying so
explicitly keeps these kernels on the CPU backend on machines that also have a
GPU, where the CUDA backend would otherwise win:


In [1]:
import triton
from triton.backends.triton_shared.driver import CPUDriver

triton.runtime.driver.set_active(CPUDriver())

In [2]:
import os
import re
import subprocess
import tempfile
from pathlib import Path

import torch

import triton.language as tl


@triton.jit
def matmul_i8_plain(a, b, c):
    offs_m = tl.arange(0, 4)
    offs_n = tl.arange(0, 4)
    offs_k = tl.arange(0, 8)
    av = tl.load(a + offs_m[:, None] * 8 + offs_k[None, :])
    bv = tl.load(b + offs_k[:, None] * 4 + offs_n[None, :])
    acc = tl.dot(av, bv, out_dtype=tl.int32)
    tl.store(c + offs_m[:, None] * 4 + offs_n[None, :], acc)


@triton.jit
def matmul_i8_ime(a, b, c):
    offs_m = tl.arange(0, 4)
    offs_n = tl.arange(0, 4)
    offs_k = tl.arange(0, 8)
    av = tl.load(a + offs_m[:, None] * 8 + offs_k[None, :])
    bv = tl.load(b + offs_k[:, None] * 4 + offs_n[None, :])
    acc = tl.dot(av, bv, out_dtype=tl.int32)
    tl.store(c + offs_m[:, None] * 4 + offs_n[None, :], acc)


@triton.jit
def matmul_f16_ime(a, b, c):
    offs_m = tl.arange(0, 4)
    offs_n = tl.arange(0, 4)
    offs_k = tl.arange(0, 8)
    av = tl.load(a + offs_m[:, None] * 8 + offs_k[None, :])
    bv = tl.load(b + offs_k[:, None] * 4 + offs_n[None, :])
    acc = tl.dot(av, bv, out_dtype=tl.float16)
    tl.store(c + offs_m[:, None] * 4 + offs_n[None, :], acc)

## Compile the two int8 variants

Both compile a 4x8 by 8x4 int8 matmul that accumulates in int32 -- the shape
and the accumulator the IME tile wants -- and only differ in the flag.


In [3]:
def compile_kernel(kernel, a, b, c, use_ime: bool):
    os.environ["TRITON_RISCV_USE_IME"] = "1" if use_ime else "0"
    return kernel.warmup(a, b, c, grid=(1,))


def readelf(kernel, flag: str) -> str:
    return disassemble(kernel, ["llvm-readelf", flag])


def disassemble(kernel, command=("llvm-objdump", "-d")) -> str:
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "kernel.o"
        path.write_bytes(kernel.asm["obj"])
        return subprocess.check_output([*command, str(path)], text=True)


def i8_inputs():
    a = torch.randint(-8, 8, (4, 8), dtype=torch.int8)
    b = torch.randint(-8, 8, (8, 4), dtype=torch.int8)
    return a, b, torch.empty((4, 4), dtype=torch.int32)


plain = compile_kernel(matmul_i8_plain, *i8_inputs(), use_ime=False)
ime = compile_kernel(matmul_i8_ime, *i8_inputs(), use_ime=True)

The object is a RISC-V relocatable file in both cases. The IME build also
records the `xsmtime` extension, which is where the matrix instructions and the
intrinsics behind them come from:


In [4]:
for label, kernel in (("plain", plain), ("ime", ime)):
    header = readelf(kernel, "-h")
    machine = re.search(r"Machine:\s+(.*)", header).group(1).strip()
    vmadot = len(re.findall(r"\bvmadot\b", disassemble(kernel)))
    print(f"{label:5} machine={machine!r} vmadot={vmadot}")

plain machine='RISC-V' vmadot=0
ime   machine='RISC-V' vmadot=1


In [5]:
recorded = [
    line.split()[-1] for line in readelf(ime, "-s").splitlines() if "xsmtime" in line
]
print(recorded[0][:150])

$xrv64i2p1_m2p0_a2p1_f2p2_d2p2_c2p0_v1p0_zicsr2p0_zmmul1p0_zaamo1p0_zalrsc1p0_zfh1p0_zfhmin1p0_zca1p0_zcd1p0_zba1p0_zbb1p0_zve32f1p0_zve32x1p0_zve64d1


`vmadot` only shows up once the flag is on.


In [6]:
assert not re.search(r"\bvmadot\b", disassemble(plain))
assert re.search(r"\bvmadot\b", disassemble(ime))
assert "xsmtime" in readelf(ime, "-s")
assert "RISC-V" in readelf(ime, "-h")
assert "vsetvli" in disassemble(ime)

## fp16 uses `vfmadot`

The same 4x8 by 8x4 shape with fp16 inputs and an fp16 accumulator lowers to
`vfmadot` instead -- the fp16 flavour of the same instruction.


In [7]:
def f16_inputs():
    a = torch.randn(4, 8, dtype=torch.float16)
    b = torch.randn(8, 4, dtype=torch.float16)
    return a, b, torch.empty((4, 4), dtype=torch.float16)


fp16 = compile_kernel(matmul_f16_ime, *f16_inputs(), use_ime=True)
print(
    {
        op: len(re.findall(rf"\b{op}\b", disassemble(fp16)))
        for op in ("vfmadot", "vmadot", "vsetvli")
    }
)

{'vfmadot': 1, 'vmadot': 0, 'vsetvli': 2}


In [8]:
assert re.search(r"\bvfmadot\b", disassemble(fp16))
assert not re.search(r"\bvmadot\b", disassemble(fp16))